# 8.5 Vector 性能（下）：DN 与 VF 建模

本节回答两个问题：**什么时候可以让数据绕过 Vector 直进 Cube（DN）**，以及**一段 VF 代码不跑能不能估出耗时（VF 建模）**。

## 1. 先看数据走 Vector 的代价

Cube 只认 NZ 分形排布，而 FA 的输入输出（Q/K/V/O）在 GM 上几乎都是 ND 排布。于是常规通路是：

```
ND 通路（Vector 中转）:
GM --MTE2--> UB --Vector: TransData(ND2NZ)--> UB --MTE3--> L1 --MTE1--> L0(Cube)

DN 通路（直进 Cube）:
GM --MTE1(ND模式)--> L0(Cube)      ← Vector 全程不碰这块数据
```

ND 通路里 Vector 干的活（transdata 转排布）**不产生任何算法价值**——纯粹是给 Cube 喂「它认识的格式」。DN 的思路就是：既然 Cube 能直接以 ND 模式取数，这步转换干脆省掉。

## 2. DN 不是免费午餐：适用条件苛刻

工程上 DN 有明确的门槛（来自实际 FA 代码的走 Dn 判定）：

| 条件 | 说明 |
|--|--|
| 输入类型受限 | float / fp8 / hifloat8 等基础类型 |
| **无可选输入** | 无位置编码（pse）、无 attention mask、无 drop mask |
| 布局对齐 | d 维模板对齐到 128 等 |
| S2 场景 | S2 有切分（序列较长）时 DN 才划算 |

**为什么 mask 会挡住 DN**：mask 需要在 mm1 之前把掩码值叠加到 QKᵀ 的输入上，这必须在 Vector 完成——既然 Vector 反正要碰这块数据，直进 Cube 的收益就没了。

**经验法则**：无可选输入且 S2 较长（有 S2 切分）→ DN；S2 很短或必须加 mask/pse → ND（Vector 中转）反而更好。

## 3. 图示：两条通路对比

![DN vs ND 数据通路](images/dn_vs_nd.svg)

## 4. DN vs ND：一笔带宽与延迟的账

| 维度 | ND（Vector 中转） | DN（直进） |
|--|--|--|
| Vector 负载 | transdata 占用大量 VF 指令 | 零 |
| GM→L0 通路 | 两跳（GM→UB→L1→L0） | 一跳（GM→L0） |
| 取数对齐要求 | 转排布后天然对齐 | ND 模式要求地址/形状满足对齐约束 |
| 适用范围 | 任意输入组合 | 受 2 节条件限制 |

**选型判据一句话**：数据「干净」（无 mask/pse/drop）、形状规整、S2 长 → DN；否则 ND。

## 动手实验：DN vs ND 建模选型


In [ ]:
# 动手：DN vs ND 通路耗时建模（抽象周期）
import numpy as np

def model_nd(s2, m, n, d):
    # ND: Vector 要做一次 ND2NZ transdata（每个元素一次读+一次写）
    transdata_cost = (m * s2 + s2 * d) * 2 * 0.05   # Q 转排布 + V 转排布
    two_hop = (m * s2 + s2 * d) * 0.08              # GM->UB->L1 两跳搬运
    return transdata_cost + two_hop

def model_dn(s2, m, n, d):
    # DN: 直进一跳, 但对齐要求带来少量填充开销
    one_hop = (m * s2 + s2 * d) * 0.05              # GM->L0 一跳
    align_pad = 0.10 * one_hop                      # ND 模式对齐填充
    return one_hop + align_pad

m, n, d = 128, 128, 128
print(f"{'S2':>6} {'ND通路':>10} {'DN通路':>10} {'更优':>6}")
for s2 in [256, 512, 1024, 2048, 4096, 8192]:
    t_nd, t_dn = model_nd(s2, m, n, d), model_dn(s2, m, n, d)
    better = 'DN' if t_dn < t_nd else 'ND'
    print(f"{s2:>6} {t_nd:>10.0f} {t_dn:>10.0f} {better:>6}")
print("\n>>> S2 越长（搬运占比越高），DN 一跳通路的优势越明显")


### 观察结论

序列越长，DN 省掉的那一整段 Vector 转排布 + 两跳搬运的收益越大。反过来说，S2 很短时 transdata 本来就没多少活，ND 通路反而因为不用满足 DN 的苛刻对齐而更稳。

## 5. VF 建模：不跑代码，数指令估耗时

VF（Vector Function）的耗时可以**静态估**，因为 VF 是顺序执行、无 Cache、无分支惩罚：

```
T_VF = Σ (每条指令的发射延迟) + Σ (相邻依赖指令间的等待)
```

其中发射延迟对每类指令是固定查表值（Load/Store/乘加/超越函数各一档）。建模流程：

1. 把 vLoop 展开：`repeatTimes = ceil(元素数 / 64)`；
2. 每轮数指令条数（优化后！）；
3. 查表累加延迟；
4. 有依赖等待的位置（写后读）加一档等待周期。

## 动手实验：Softmax VF 建模


In [ ]:
# 动手：Softmax VF 的耗时建模
import numpy as np

S1, S2 = 128, 128          # 基本块
REG_ELEMS = 64             # 每条 VF 指令 64 个 FP32

# 指令延迟表（抽象周期, 示意值）
LAT = {'load': 4, 'store': 4, 'mul': 2, 'add': 2,
       'max': 3, 'exp': 8, 'div': 12, 'dup': 2}

# Softmax 沿 S2 方向归约, 每行 S2 个元素, 共 S1 行
rows = S1
elems_per_row = -(-S2 // REG_ELEMS)     # 每行的 VF 轮数
rounds = rows * elems_per_row

# 每轮的指令清单（已应用 8.4 的优化: 常量外提）
per_round = [
    ('load', 1),      # 载入当前行数据块
    ('max', 1),       # 行最大值（归约到标量寄存器）
    ('exp', 1),       # exp(x - max)
    ('add', 1),       # 行求和（归约）
    ('div', 1),       # 归一化除法
    ('store', 1),     # 写回
]

t_per_round = sum(LAT[op] * cnt for op, cnt in per_round)
t_total = t_per_round * rounds
# 相邻轮的写后读等待
t_wait = rounds * 2

print(f"基本块 S1={S1}, S2={S2}, 每行 VF 轮数={elems_per_row}, 总轮数={rounds}")
print(f"每轮指令: {[op for op, _ in per_round]}")
print(f"每轮周期: {t_per_round}, 建模总周期: {t_total + t_wait}")
print(f"理论下限(纯计算): {rounds * (LAT['exp'] + LAT['div'])}")
print(f">>> 搬运+归约占了 {(1 - (LAT['exp']+LAT['div'])/t_per_round):.0%} 的每轮开销——优化方向一目了然")


### 观察结论

建模把「感觉慢」变成「哪条指令贵」：上例中 exp/div 只占每轮周期的一小部分，**大头在搬运和归约的配合**上。这正是 8.4 说的「指令编排」要解决的问题——建模先行，动手才有方向。

## 6. 建模的工程用法

1. **写代码前**：估 VF 方案的耗时上限，与 Vector 总预算对比，不够就换方案（比如走 DN）；
2. **写完后**：实测 vs 建模对比，偏差大说明实现有问题（bank 冲突、依赖等待没编排好）；
3. **调优时**：建模指出最贵的指令类，优先砍它。

## 本节要点

- DN = 数据以 ND 排布直进 Cube，省掉 Vector 的 transdata 与一跳搬运；
- DN 门槛苛刻：无可选输入（mask/pse/drop）、类型/对齐受限、S2 较长才划算；
- 选型口诀：**数据干净 + S2 长 → DN；要加 mask 或 S2 短 → ND**；
- VF 耗时可静态建模：数指令 × 查表延迟 + 依赖等待，先建模再动手。

## 小测验

1. 画出 ND 与 DN 两条通路的数据流（从 GM 到 L0），标出 Vector 的参与点。
2. 为什么带 attention mask 的场景 DN 不划算？
3. VF 建模的公式是什么？为什么 VF 可以静态建模而 CPU 代码很难？
4. 建模发现某 VF 函数中 Load/Store 占了 70% 周期，下一步优化方向是什么？

>>> 参考答案见 [answer/08.05_answer.txt](answer/08.05_answer.txt)
